# Foco-LLM: intervenção em notas de raciocínio (7B)

Teste exploratório pareado em 20 questões do holdout fora dos moldes. Todos os fatos de entrada permanecem iguais; apenas linhas de uma nota escrita pelo modelo são removidas. A nota deve obedecer ao formato pré-especificado. Casos sem nota válida são rejeitados e relatados, nunca contados como acertos ou erros de seleção.


In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ["MPLBACKEND"] = "Agg"
if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError(f"Python 3.12 or 3.13 required; found {sys.version}")
if shutil.which("poetry") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "poetry==2.4.1"], check=True)
repository = Path("/kaggle/temp/foco-llm-note-intervention")
output = Path("/kaggle/working/note-intervention-7b")
subprocess.run(
    [
        "git",
        "clone",
        "--branch",
        "feat/deduction-specialist-7b",
        "--single-branch",
        "https://github.com/Mathwesm/foco-llm.git",
        str(repository),
    ],
    check=True,
)
source_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=repository, text=True
).strip()
subprocess.run(
    ["poetry", "install", "--only", "main,inference", "--no-interaction"],
    cwd=repository,
    check=True,
)
print("Source commit:", source_commit, flush=True)

In [ ]:
holdout = output / "holdout"
subprocess.run(
    ["poetry", "run", "python", "scripts/run_ood_chain_holdout.py", "--output", str(holdout)],
    cwd=repository,
    check=True,
)
dataset = holdout / "dataset.json"
manifest = json.loads((holdout / "manifest.json").read_text(encoding="utf-8"))
expected_hash = manifest["dataset_sha256"]
if (
    hashlib.sha256(dataset.read_bytes()).hexdigest() != expected_hash
    or manifest["dataset_sha256"] != expected_hash
):
    raise ValueError("Frozen OOD dataset hash mismatch")
print("Frozen dataset verified:", expected_hash, flush=True)

In [ ]:
subprocess.run(
    [
        "poetry",
        "run",
        "python",
        "scripts/run_reasoning_note_intervention.py",
        "--dataset",
        str(dataset),
        "--output",
        str(output / "base"),
        "--backend",
        "cloud",
    ],
    cwd=repository,
    check=True,
)
summaries = sorted((output / "base").glob("*/summary.json"))
if len(summaries) != 1:
    raise RuntimeError(f"Expected one evaluation summary; found {len(summaries)}")
result = json.loads(summaries[0].read_text(encoding="utf-8"))
comparison = {
    "protocol": "model-written-note-intervention-v1",
    "source_commit": source_commit,
    "dataset_sha256": expected_hash,
    "result": result,
}
(output / "comparison.json").write_text(json.dumps(comparison, indent=2), encoding="utf-8")
print(json.dumps(comparison, indent=2), flush=True)